**AlphaGenome profiling: tasks**
Run top to bottom. Weights and genome go to the VM disk (`/content/ag_data`), results to `experiments/results/`. One section per task, each one script; rerun only the section you changed.

**1. Setup: pull the branch and install**

In [ ]:
%env AG_DATA=/content/ag_data
%env HF_HOME=/content/ag_data/hf
!git clone -q -b experiments-profiling https://github.com/Shivaritha22/alphagenome-pytorch /content/ag 2>/dev/null || git -C /content/ag pull -q
%cd /content/ag/experiments
!pip install -q -e .. pyfaidx huggingface_hub
!git log -1 --oneline
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

**2. Weights (download once per session)**

In [ ]:
!python -c "import common; p = common.fetch_weights(); print(p, p.stat().st_size >> 20, 'MB')"

**3. Reference genome (download once per session; only task 4 needs it)**

In [ ]:
!mkdir -p $AG_DATA/ref
!test -f $AG_DATA/ref/GRCh38.p13.genome.fa || wget -q -O $AG_DATA/ref/GRCh38.p13.genome.fa https://storage.googleapis.com/alphagenome/reference/gencode/hg38/GRCh38.p13.genome.fa
!test -s $AG_DATA/ref/GRCh38.p13.genome.fa.fai || wget -q -O $AG_DATA/ref/GRCh38.p13.genome.fa.fai https://storage.googleapis.com/alphagenome/reference/gencode/hg38/GRCh38.p13.genome.fa.fai
!ls -lh $AG_DATA/ref

**4. Task 4: fixed input**
Cuts `chr19:40480552-41529128` (1 Mb centred on CYP2B6) into `inputs/cyp2b6_1m.json` + `cyp2b6_1m_onehot.npy`.
Want: `N bases: 0` and `131k crop matches jax run: True`. Commit the `.json`; the `.npy` stays out of git and is rebuilt here after a disconnect.

In [ ]:
!python input.py
!cat inputs/cyp2b6_1m.json

**5. Task 3: forward pass at every length**
Want: one `ok` line per length, then `all checks passed`. The centre-r table is reported, not pass/fail: r should approach 1 as the shorter input gets closer to the longest. Add `-v` to print every output's dtype and shape.

In [ ]:
!python lengths.py